<a href="https://colab.research.google.com/github/ManideepLadi/cs6910_assignment3/blob/manideep/RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Dakshina Dataset from google


In [5]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


In [6]:
!wget https://storage.googleapis.com/gresearch/dakshina/dakshina_dataset_v1.0.tar

--2021-05-03 06:27:45--  https://storage.googleapis.com/gresearch/dakshina/dakshina_dataset_v1.0.tar
Resolving storage.googleapis.com (storage.googleapis.com)... 173.194.76.128, 66.102.1.128, 172.253.120.128, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|173.194.76.128|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2008340480 (1.9G) [application/x-tar]
Saving to: ‘dakshina_dataset_v1.0.tar’

dakshina_dataset_v1 100%[===================>]   1.87G   134MB/s    in 21s     

2021-05-03 06:28:07 (89.6 MB/s) - ‘dakshina_dataset_v1.0.tar’ saved [2008340480/2008340480]



In [7]:
!tar -xvf '/content/dakshina_dataset_v1.0.tar'

dakshina_dataset_v1.0/bn/
dakshina_dataset_v1.0/bn/lexicons/
dakshina_dataset_v1.0/bn/lexicons/bn.translit.sampled.test.tsv
dakshina_dataset_v1.0/bn/lexicons/bn.translit.sampled.train.tsv
dakshina_dataset_v1.0/bn/lexicons/bn.translit.sampled.dev.tsv
dakshina_dataset_v1.0/bn/native_script_wikipedia/
dakshina_dataset_v1.0/bn/native_script_wikipedia/bn.wiki-filt.valid.text.shuf.txt.gz
dakshina_dataset_v1.0/bn/native_script_wikipedia/bn.wiki-full.info.sorted.tsv.gz
dakshina_dataset_v1.0/bn/native_script_wikipedia/bn.wiki-filt.train.info.sorted.tsv.gz
dakshina_dataset_v1.0/bn/native_script_wikipedia/bn.wiki-filt.train.text.sorted.tsv.gz
dakshina_dataset_v1.0/bn/native_script_wikipedia/bn.wiki-filt.train.text.shuf.txt.gz
dakshina_dataset_v1.0/bn/native_script_wikipedia/bn.wiki-full.nonblock.sections.tsv.gz
dakshina_dataset_v1.0/bn/native_script_wikipedia/bn.wiki-full.omit_pages.txt.gz
dakshina_dataset_v1.0/bn/native_script_wikipedia/bn.wiki-full.text.sorted.tsv.gz
dakshina_dataset_v1.0/bn/na

Preprocess data

In [8]:
# import string
 
# # load doc into memory
# def load_doc(filename):
# 	# open the file as read only
# 	file = open(filename, mode='rt', encoding='utf-8')
# 	# read all text
# 	text = file.read()
# 	# close the file
# 	file.close()
# 	return text
 
# # split a loaded document into sentences
# def to_pairs(doc):
# 	lines = doc.strip().split('\n')
# 	pairs = [line.split('\t') for line in  lines]
# 	return pairs


In [9]:

# load dataset
filename = 'dakshina_dataset_v1.0/te/lexicons/te.translit.sampled.train.tsv'
# doc = load_doc(filename)
# # split into english-german pairs
# pairs = to_pairs(doc)

In [10]:
# # Vectorize the data.
# input_characters = set()
# target_characters = set()
# for pair in pairs:
#   for char in pair[1]:
#     if pair[1] not in input_characters:
#       input_characters.add(char)
#   for char in pair[0]:
#     if char not in target_characters:
#       target_characters.add(char)

# input_characters = sorted(list(input_characters))
# target_characters = sorted(list(target_characters))
# num_encoder_tokens = len(input_characters)
# num_decoder_tokens = len(target_characters)


# print("Number of unique input tokens:", num_encoder_tokens)
# print("Number of unique output tokens:", num_decoder_tokens)


In [11]:
# Vectorize the data.
input_texts = []
target_texts = []
input_characters = set()
target_characters = set()
with open(filename, "r", encoding="utf-8") as f:
    lines = f.read().split("\n")
for line in lines[: len(lines) - 1]:
    target_text,input_text, attestation = line.split("\t")
    # We use "tab" as the "start sequence" character
    # for the targets, and "\n" as "end sequence" character.
    target_text = "\t" + target_text + "\n"
    for i in range(int(attestation)):
      input_texts.append(input_text)
      target_texts.append(target_text)
    for char in input_text:
        if char not in input_characters:
            input_characters.add(char)
    for char in target_text:
        if char not in target_characters:
            target_characters.add(char)

input_characters = sorted(list(input_characters))
target_characters = sorted(list(target_characters))
num_encoder_tokens = len(input_characters)
num_decoder_tokens = len(target_characters)
max_encoder_seq_length = max([len(txt) for txt in input_texts])
max_decoder_seq_length = max([len(txt) for txt in target_texts])

print("Number of samples:", len(input_texts))
print("Number of unique input tokens:", num_encoder_tokens)
print("Number of unique output tokens:", num_decoder_tokens)
print("Max sequence length for inputs:", max_encoder_seq_length)
print("Max sequence length for outputs:", max_decoder_seq_length)

Number of samples: 84680
Number of unique input tokens: 26
Number of unique output tokens: 65
Max sequence length for inputs: 25
Max sequence length for outputs: 22


In [12]:
input_texts[1]

'ankita'

In [13]:
target_texts[1]

'\tఅంకిత\n'

In [14]:
input_characters[3]

'd'

In [15]:
input_token_index = dict([(char, i) for i, char in enumerate(input_characters)])
target_token_index = dict([(char, i) for i, char in enumerate(target_characters)])

reverse_input_char_index = dict((i, char) for char, i in input_token_index.items())
reverse_target_char_index = dict((i, char) for char, i in target_token_index.items())

encoder_input_data = np.zeros(
    (len(input_texts), max_encoder_seq_length, num_encoder_tokens), dtype="float32"
)
decoder_input_data = np.zeros(
    (len(input_texts), max_decoder_seq_length, num_decoder_tokens), dtype="float32"
)
decoder_target_data = np.zeros(
    (len(input_texts), max_decoder_seq_length, num_decoder_tokens), dtype="float32"
)

for i, (input_text, target_text) in enumerate(zip(input_texts, target_texts)):
    for t, char in enumerate(input_text):
        encoder_input_data[i, t, input_token_index[char]] = 1.0
    for t, char in enumerate(target_text):
        # decoder_target_data is ahead of decoder_input_data by one timestep
        decoder_input_data[i, t, target_token_index[char]] = 1.0
        if t > 0:
            # decoder_target_data will be ahead by one timestep
            # and will not include the start character.
            decoder_target_data[i, t - 1, target_token_index[char]] = 1.0

In [16]:
target_texts[0]

'\tఅంకిత\n'

In [17]:
encoder_input_data[0,6]

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0.], dtype=float32)

In [18]:
decoder_input_data[0,6]

array([0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
      dtype=float32)

In [19]:
num_encoder_tokens

26

In [49]:
class RNN_Model:

  def __init__(self,input_embedding_size,no_of_encoder_layers,no_of_decoder_layers,latent_dimension,dropout,recurrent_dropout,beam_size,cell_type):
    self.input_embedding_size = input_embedding_size
    self.no_of_encoder_layers = no_of_encoder_layers
    self.no_of_decoder_layers = no_of_decoder_layers
    self.latent_dimension = latent_dimension
    self.dropout = dropout
    self.recurrent_dropout=recurrent_dropout
    self.beam_size = beam_size
    self.cell_type=cell_type
    self.model = None

  def printModelParameters(self):
    print(self.input_embedding_size)
    print(self.no_of_encoder_layers)
    print(self.no_of_decoder_layers)
    print(self.latent_dimension)
    print(self.dropout)
    print(self.model.summary())

  def BUILD_MODEL(self,num_encoder_tokens,num_decoder_tokens):
    encoder_inputs = keras.Input(shape=(None, num_encoder_tokens))

    outputs = encoder_inputs
    encoder_states = []
    for j in range(self.no_of_encoder_layers)[::-1]:
      if self.cell_type == "LSTM":
        outputs, h , c = keras.layers.LSTM(self.latent_dimension, return_state=True, return_sequences=bool(j),dropout=self.dropout,recurrent_dropout=self.recurrent_dropout)(outputs)
        encoder_states += [h,c]
      elif self.cell_type == "GRU" :
        outputs, h = keras.layers.GRU(self.latent_dimension, return_state=True, return_sequences=bool(j),dropout=self.dropout,recurrent_dropout=self.recurrent_dropout)(outputs)
        encoder_states += [h]
      elif self.cell_type == "RNN" :
        outputs, h = keras.layers.SimpleRNN(self.latent_dimension, return_state=True, return_sequences=bool(j),dropout=self.dropout,recurrent_dropout=self.recurrent_dropout)(outputs)
        encoder_states += [h]

    decoder_inputs = keras.Input(shape=(None, num_decoder_tokens))

    outputs = decoder_inputs
    output_layers = []
    for j in range(self.no_of_decoder_layers):
        if self.cell_type == "LSTM":
          output_layers.append(
              keras.layers.GRU(self.latent_dimension, return_sequences=True, return_state=True,dropout=self.dropout,recurrent_dropout=self.recurrent_dropout)
          )
          outputs, dh, dc = output_layers[-1](outputs, initial_state=[encoder_states[-2],encoder_states[-1]])
        elif self.cell_type == "GRU" : 
          output_layers.append(
              keras.layers.GRU(self.latent_dimension, return_sequences=True, return_state=True,dropout=self.dropout,recurrent_dropout=self.recurrent_dropout)
          )
          outputs, dh = output_layers[-1](outputs, initial_state=encoder_states[-1])
        elif self.cell_type == "RNN" : 
          output_layers.append(
              keras.layers.SimpleRNN(self.latent_dimension, return_sequences=True, return_state=True,dropout=self.dropout,recurrent_dropout=self.recurrent_dropout)
          )
          outputs, dh = output_layers[-1](outputs, initial_state=encoder_states[-1])


    decoder_dense = keras.layers.Dense(num_decoder_tokens, activation='softmax')
    decoder_outputs = decoder_dense(outputs)
    # Define the model that will turn
    # `encoder_input_data` & `decoder_input_data` into `decoder_target_data`
    self.model = keras.Model([encoder_inputs, decoder_inputs], decoder_outputs)
    self.model.compile(optimizer='rmsprop',loss='categorical_crossentropy',metrics=['accuracy']) 
    return


  #Fit model using the train datagenerator and returns fitted model..
  def FIT_RNN(self , encoder_input_data,decoder_input_data ,decoder_target_data,epochs ,batch_size):
    self.model.fit(
        [encoder_input_data, decoder_input_data],
        decoder_target_data,
        batch_size=batch_size,
        epochs=epochs,
        validation_split=0.1)
        # validation_split=0.1,
        # callbacks = [WandbCallback(monitor='val_accuracy',
        #                                             save_model = True))
    return



In [48]:
rnn = RNN_Model(32,3,3,64,0.3,0,0,"GRU")
rnn.BUILD_MODEL(num_encoder_tokens,num_decoder_tokens)
rnn.printModelParameters()
rnn.FIT_RNN( encoder_input_data,decoder_input_data,  decoder_target_data,
    batch_size=64,
    epochs=10)

32
3
3
64
0.3
Model: "model_4"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_13 (InputLayer)           [(None, None, 26)]   0                                            
__________________________________________________________________________________________________
gru_20 (GRU)                    [(None, None, 64), ( 17664       input_13[0][0]                   
__________________________________________________________________________________________________
gru_21 (GRU)                    [(None, None, 64), ( 24960       gru_20[0][0]                     
__________________________________________________________________________________________________
input_14 (InputLayer)           [(None, None, 65)]   0                                            
______________________________________________________________________________

In [50]:
!pip install wandb -qqq
import wandb
wandb.login()

     |████████████████████████████████| 2.1MB 11.7MB/s 
     |████████████████████████████████| 133kB 42.9MB/s 
     |████████████████████████████████| 163kB 41.9MB/s 
     |████████████████████████████████| 102kB 11.7MB/s 
     |████████████████████████████████| 71kB 9.2MB/s 


<IPython.core.display.Javascript object>

wandb: You can find your API key in your browser here: https://wandb.ai/authorize


wandb: Paste an API key from your profile and hit enter: ··········


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [51]:
sweep_config = {
  'name': 'RNN',
  'method': 'grid',
  'metric': {
      'name': 'accuracy',
      'goal': 'maximize'   
    },
  'parameters': {
        'input_embedding_size': {
            'values': [16, 32, 64, 256]
        },
        'encoder_layers':{
            'values':[1,2,3]
        },
        'decoder_layers':{
            'values':[1,2,3]
        },
        'hidden_layer_size':{
            'values':[16, 32, 64, 256]
        },
        'cell_type':{
            'values':['RNN', 'GRU', 'LSTM']
        },
        'dropout':{
            'values':[0.3,0.2,0.0]
        },
        'recurrent_dropout':{
            'values':[0.3,0.2,0.0]
        },
        'beam_sizes':{
            'values':['No','Yes']
        }

    }
}

sweep_id = wandb.sweep(sweep_config, project='RNN', entity='manideepladi')

Create sweep with ID: dr6rhbrm
Sweep URL: https://wandb.ai/manideepladi/RNN/sweeps/dr6rhbrm


In [ ]:
def train():
  run = wandb.init()
  configuration=run.config

  rnn = RNN_Model(input_embedding_size=configuration.input_embedding_size,no_of_encoder_layers=configuration.encoder_layers,no_of_decoder_layers=configuration.decoder_layers,
                  latent_dimension=configuration.hidden_layer_size,dropout=configuration.dropout,recurrent_dropout=configuration.recurrent_dropout,beam_size=configuration.beam_sizes,cell_type=configuration.cell_type)
  rnn.BUILD_MODEL(num_encoder_tokens,num_decoder_tokens)
  rnn.printModelParameters()
  rnn.FIT_RNN( encoder_input_data,decoder_input_data,  decoder_target_data,
    batch_size=64,
    epochs=10)
#train()
wandb.agent(sweep_id=sweep_id, function=train)

wandb: Agent Starting Run: fstju9ew with config:
wandb: 	beam_sizes: No
wandb: 	cell_type: RNN
wandb: 	decoder_layers: 1
wandb: 	dropout: 0.3
wandb: 	encoder_layers: 1
wandb: 	hidden_layer_size: 16
wandb: 	input_embedding_size: 64
wandb: 	recurrent_dropout: 0.3


64
1
1
16
0.3
Model: "model"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, None, 26)]   0                                            
__________________________________________________________________________________________________
input_2 (InputLayer)            [(None, None, 65)]   0                                            
__________________________________________________________________________________________________
simple_rnn (SimpleRNN)          [(None, 16), (None,  688         input_1[0][0]                    
__________________________________________________________________________________________________
simple_rnn_1 (SimpleRNN)        [(None, None, 16), ( 1312        input_2[0][0]                    
                                                                 simple_rnn[0][1

In [42]:
print(rnn.model.layers[8].name)

dense_2


In [22]:
latent_dims = [1024, 1024,  1024]
for j in range(3)[::-1]:
  print(j)



2
1
0


In [20]:
encoder_input_data.shape

(84680, 25, 26)

In [21]:
decoder_input_data.shape

(84680, 22, 65)

In [22]:
decoder_target_data.shape


(84680, 22, 65)

In [ ]:
encoder_model = keras.Model(encoder_inputs, encoder_states)


d_outputs = decoder_inputs
decoder_states_inputs = []
decoder_states = []
for j in range(len(latent_dims))[::-1]:
    current_state_inputs = [keras.Input(shape=(latent_dims[j],)) for _ in range(2)]

    temp = output_layers[len(latent_dims)-j-1](d_outputs, initial_state=current_state_inputs)

    d_outputs, cur_states = temp[0], temp[1:]

    decoder_states += cur_states
    decoder_states_inputs += current_state_inputs

decoder_outputs = decoder_dense(d_outputs)
decoder_model = keras.Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs] + decoder_states)


def decode_sequence(input_seq, encoder_model, decoder_model):
    # Encode the input as state vectors.
    states_value = encoder_model.predict(input_seq)

    # Generate empty target sequence of length 1.
    target_seq = np.zeros((1, 1, num_decoder_tokens))
    # Populate the first character of target sequence with the start character.
    target_seq[0, 0, target_token_index['\t']] = 1.

    # Sampling loop for a batch of sequences
    # (to simplify, here we assume a batch of size 1).
    stop_condition = False
    decoded_sentence = []  #Creating a list then using "".join() is usually much faster for string creation
    while not stop_condition:
        to_split = decoder_model.predict([target_seq] + states_value)

        output_tokens, states_value = to_split[0], to_split[1:]

        # Sample a token
        sampled_token_index = np.argmax(output_tokens[0, 0])
        sampled_char = reverse_target_char_index[sampled_token_index]
        decoded_sentence.append(sampled_char)

        # Exit condition: either hit max length
        # or find stop character.
        if sampled_char == '\n' or len(decoded_sentence) > max_decoder_seq_length:
            stop_condition = True

        # Update the target sequence (of length 1).
        target_seq = np.zeros((1, 1, num_decoder_tokens))
        target_seq[0, 0, sampled_token_index] = 1.

    return "".join(decoded_sentence)

In [53]:
for seq_index in range(200):
    # Take one sequence (part of the training set)
    # for trying out decoding.
    input_seq = encoder_input_data[seq_index +100: seq_index + 101]
    decoded_sentence = decode_sequence(input_seq,encoder_model,decoder_model)
    print("-")
    print("Input sentence:", input_texts[seq_index])
    print("Decoded sentence:", decoded_sentence)

-
Input sentence: amkita
Decoded sentence: అంచనాలతో

-
Input sentence: ankita
Decoded sentence: అంచనలను

-
Input sentence: ankita
Decoded sentence: అంచనాలను

-
Input sentence: ankitha
Decoded sentence: అంచనాలను

-
Input sentence: ankitha
Decoded sentence: అంచనాలు

-
Input sentence: ankitam
Decoded sentence: అంచనాలు

-
Input sentence: ankitham
Decoded sentence: అంచనాలు

-
Input sentence: ankitham
Decoded sentence: అంచనాలు

-
Input sentence: ankitabaavam
Decoded sentence: అంచు

-
Input sentence: ankithabhavam
Decoded sentence: అంచు

-
Input sentence: ankithabhavam
Decoded sentence: అంచు

-
Input sentence: ankatamicchaadu
Decoded sentence: అంచున

-
Input sentence: ankitamicchadu
Decoded sentence: అంచున

-
Input sentence: ankitamichhaadu
Decoded sentence: అంచున

-
Input sentence: ankitamichhaadu
Decoded sentence: అంచుని

-
Input sentence: ankithamicchaadu
Decoded sentence: అంచుని

-
Input sentence: ankithamichhaadu
Decoded sentence: అంచుని

-
Input sentence: amkithamichaaru
Decoded sentenc